# 09 · Manual handoff: continue from files you produced yourself

**When:** the data is not reachable end to end. The Left side lives in SAS, Excel or a text dump;
the Right side is not in Databricks yet (for example only in Athena). You produce an
**intermediate extract** with your own tools, put it where the pipeline can read it, and the
pipeline continues from Stage 1 as usual.

**Principle:** hand over *rows*, never hand-made statistics. Counts, hashes and sampling must be
computed by dtrack on both sides, otherwise the two sides are not comparable.

| Side | Extract | Where it goes | How the pipeline reads it |
| --- | --- | --- | --- |
| Left | cleaned CSV (all text) or parquet | this PC, e.g. `inputs/<pair>/left/` | `source: file` (DuckDB) |
| Right | parquet from Athena `UNLOAD`, or CSV | `s3://<bucket>/dtrack/work/inputs/<pair>/right/` = `/Volumes/<cat>/<schema>/work/inputs/<pair>/right/` | `kind: files` in the Databricks job, or `handoff.FileTarget` on any machine |

The helpers used here live in `examples/nbs/handoff.py` (a prototype, not part of `src/`).

## Files you must prepare

| File | Content | Rules |
| --- | --- | --- |
| **pair config** | `left`, `right`, `col_map`, `key`, `key_families`, `sample`, `vintage`, `fromDate`/`toDate`, optional `col_type_overrides`, `date_synonyms`, `strict` | `col_map` maps **left name → right name**; key columns must be in it |
| **left extract** | one row per record, only the pair's rows | header row; UTF-8; every value as text |
| **right extract** | the same, from the Right system | same filter and period as the left one |
| `_SOURCE.json` next to each extract | tool, query, producer, time, row count, sha256 | written by `handoff.write_extract` |

## Conventions for an extract

1. **Dates**: canonical `YYYY-MM-DD` in the date column, or declare `date_format` in the pair.
   SAS numeric dates (days since 1960-01-01) and Excel serials (since 1899-12-30) must be converted.
2. **Nulls**: an empty cell. Never `.`, `NULL`, `NA`, `N/A`, `#N/A`, `None`.
3. **Numbers**: `.` as decimal point, no thousands separators, no scientific notation, no rounding.
   Trailing zeros are fine (`7.50` and `7.5` compare equal).
4. **Keys**: keep them as text. Excel drops leading zeros (`00123` → `123`); pad them back when the
   width is known.
5. **Text**: trimmed (SAS pads character columns), UTF-8.
6. **Same scope on both sides**: identical period, filters and deduplication.
7. **Different names and types are fine**: `col_map` pairs the names; the types must end up in the
   same family (numeric or text) on both sides, or be forced with `col_type_overrides`.

In [ ]:
import json

import duckdb
import handoff
import nbkit
import pandas as pd
from nbkit import show, table

## 1 · What goes wrong in a typical spreadsheet export

A small sample with every classic problem, checked before anything is uploaded.

In [ ]:
folder = nbkit.sandbox("09")
messy = pd.DataFrame(
    {
        "ORDER_DT": ["46023", "46024", "46024", "46025"],
        "ORDER_NO": ["123", "00124", "1.25E+05", "126"],
        "AMT": ["1,234.50", "7.5", ".", "20"],
        "STATUS": ["NEW ", "PAID", "N/A", "PAID"],
    }
)
report = handoff.check_handoff(
    messy, date_col="ORDER_DT", columns=["AMT", "STATUS"], key_cols=["ORDER_NO"]
)
table(report)

In [ ]:
fixed = handoff.normalize_export(
    messy.assign(ORDER_NO=messy.ORDER_NO.replace({"1.25E+05": "125000"})),
    date_col="ORDER_DT",
    date_parser=handoff.excel_date,
    thousands=["AMT"],
    zfill={"ORDER_NO": 5},
)
table(fixed)
table(
    handoff.check_handoff(
        fixed, date_col="ORDER_DT", columns=["AMT", "STATUS"], key_cols=["ORDER_NO"]
    )
)

Every error is gone. One warning is left on purpose: `125000` is wider than the padded keys.
It came from the scientific-notation key (`1.25E+05`), which cannot be repaired from the file
because the digits are gone; re-export that column as text. A checker can only flag it.

## 2 · Left: a SAS export with its own names and formats

SAS wrote `ORDER_DT` as `DATE9.` (`01JAN2026`), `ORDER_NO` as the key, amounts with three
decimals and space-padded character columns. We normalise, check, and write the extract with its
provenance.

In [ ]:
data, config, _ = nbkit.dataset(folder)
left = pd.read_csv(data / "left" / "ORDERS.csv", dtype=str)
sas = pd.DataFrame(
    {
        "ORDER_DT": pd.to_datetime(left.BUSINESS_DATE).dt.strftime("%d%b%Y").str.upper(),
        "ORDER_NO": left.ORDER_ID,
        "AMT": left.AMOUNT.astype(float).map("{:.3f}".format),
        "STATUS": left.STATUS.str.ljust(8),
        "REGION": left.REGION,
    }
)
table(sas.head(3))
table(
    handoff.check_handoff(
        sas, date_col="ORDER_DT", columns=["AMT", "STATUS", "REGION"], key_cols=["ORDER_NO"]
    )
)

In [ ]:
clean = handoff.normalize_export(sas, date_col="ORDER_DT")
left_path = handoff.write_extract(
    clean,
    folder / "inputs/orders/left/orders.csv",
    tool="SAS 9.4 PROC EXPORT",
    query="data work.orders; set fin.orders; where '01JAN2026'd <= order_dt <= '28FEB2026'd; run;",
    producer="analyst",
    note="SAS DATE9 converted to ISO, char padding trimmed",
)
table(clean.head(3))
show(json.loads(left_path.with_name("_SOURCE.json").read_text()), "_SOURCE.json")

## 3 · Right: an Athena export instead of a Databricks table

In production you run, in Athena:

```sql
UNLOAD (
  SELECT id, biz_dt, amt, status, region FROM sales.orders
  WHERE biz_dt BETWEEN DATE '2026-01-01' AND DATE '2026-02-28'
) TO 's3://dve-tools/dtrack/work/inputs/orders/right/' WITH (format = 'PARQUET')
```

The prefix is inside the work Volume, so the Databricks job reads it as
`/Volumes/dve_catlog/dve_live/work/inputs/orders/right/` with `kind: files`. Here DuckDB writes the
same kind of typed parquet file locally.

In [ ]:
right_dir = folder / "inputs/orders/right"
right_dir.mkdir(parents=True)
duckdb.sql(f"""
COPY (
  SELECT CAST(order_id AS BIGINT) AS id, CAST(business_date AS DATE) AS biz_dt,
         CAST(amount AS DECIMAL(10, 2)) AS amt, status, region
  FROM read_csv('{data / "right" / "orders.csv"}', all_varchar = true)
) TO '{right_dir / "part-0000.parquet"}' (FORMAT parquet)
""")
print(
    duckdb.sql(f"DESCRIBE SELECT * FROM '{right_dir}/part-0000.parquet'")
    .df()[["column_name", "column_type"]]
    .to_string(index=False)
)

## 4 · The pair config ties the two extracts together

Different names on each side (`ORDER_NO` ↔ `id`, `AMT` ↔ `amt`), different types (text vs
`BIGINT`/`DECIMAL`). Both end up numeric: the left by value, the right by type. `validate_config`
checks the pair before anything runs.

In [ ]:
from dtrack_left.cli.config import validate_config

pair = {
    "left": {
        "source": "file",
        "path": str(left_path),
        "date_col": "ORDER_DT",
        "date_type": "string_dash",
    },
    "right": {
        "source": "databricks",
        "kind": "files",
        "path": str(right_dir / "*.parquet"),
        "format": "parquet",
        "date_col": "biz_dt",
        "date_type": "date",
    },
    "col_map": {"ORDER_NO": "id", "AMT": "amt", "STATUS": "status", "REGION": "region"},
    "key": "ORDER_NO",
    "key_families": {"ORDER_NO": "numeric"},
    "sample": 0.25,
    "vintage": "month",
    "fromDate": "2026-01-01",
    "toDate": "2026-02-28",
}
handoff_config = validate_config({"pairs": {"orders": pair}})
print("validate_config: ok")

## 5 · Continue the pipeline from the extracts

Left counts and statistics come from the SAS extract; the Right half runs with
`handoff.FileTarget` over the parquet export (in production the same config runs as the
Databricks job with `kind: files`).

In [ ]:
from dtrack_left.checks.col_check import ColCheckPipeline, LeftRuntime, wait_for_manifests
from dtrack_right.checks.col_check import ColCheck
from dtrack_right.report import build_report

store = nbkit.VolumeStore(folder / "runs" / "handoff")
right_target = lambda config: handoff.FileTarget(config, folder / "right_work")
nbkit.left_row_check(handoff_config, store, run_id="handoff")
nbkit.right_row_check(store, right_target)
show(store.get_json("row_check/orders.json")["summary"], "Stage 1 on the extracts")
manifests = wait_for_manifests(store, timeout=1, poll_interval=0)
runtime = LeftRuntime(folder / "left_work", store)
print(
    "units:",
    ColCheckPipeline(
        store, pull_window=runtime.pull_window, compute_stats=runtime.compute_stats
    ).run(manifests),
)
results = ColCheck(store, right_target, poll_interval=0).run()
table(
    pd.DataFrame(results)[
        ["vintage", "column", "right_col", "col_type", "verdict", "hash_sum_match"]
    ].sort_values(["vintage", "column"])
)
show(
    json.loads(build_report(store, folder / "report")["summary.json"].read_text())["findings"],
    "report findings",
)

The findings are exactly the differences planted in the synthetic data (a count diff, two missing
partitions, `STATUS` lower-cased in January, `AMOUNT` +0.01 in February). Nothing came from the
format changes: `01JAN2026`, `436.020`, padded `NEW     `, `DECIMAL` and `BIGINT` all compared
equal after normalisation.

## Pitfalls

- **Hand-made statistics.** Never upload counts or summaries computed in SAS or Excel. `hash_sum`,
  sampling and decimal collapsing only agree when dtrack computes them on both sides.
- **Different scope.** A filter, period or dedup step applied to one extract only shows up as
  count diffs everywhere. Put the exact query in `_SOURCE.json`.
- **Text vs number.** If a column is numeric on one side and categorical on the other (for example
  the Right export is all strings), the verdict is `diff` even when the values agree. Use a typed
  export or `col_type_overrides` (notebook 04).
- **Unrecoverable exports.** Scientific notation and lost leading zeros cannot always be repaired.
  Export as text at the source.
- **Right extract outside the Volume.** The Databricks job can only read what the work Volume (or
  another external location) covers; `inputs/` under the work prefix needs no new permission.